In [ ]:
##importing env variables
import os
from dotenv import load_dotenv
load_dotenv()



True

In [3]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [5]:
#DATA INGESTION

from langchain_community.document_loaders import WebBaseLoader

In [ ]:
###WRAP DRIVE ARTICLE   
webloader=WebBaseLoader("https://www.ebsco.com/research-starters/engineering/warp-drive")

In [8]:
res=webloader.load()
res

[Document(metadata={'source': 'https://www.ebsco.com/research-starters/engineering/warp-drive', 'title': 'Warp drive | Engineering | Research Starters | EBSCO Research', 'description': '<p>Warp drive is a theoretical propulsion method that enables spacecraft to exceed the speed of light, primarily explored in science fiction literature and media. Originating in the 1920s and gaining prominence through the iconic television series Star Trek, the warp drive concept serves as a solution to the immense distances in the universe that hinder space travel. It proposes the creation of a &quot;warped&quot; bubble around a spacecraft, facilitated by a substantial power source, allowing it to navigate faster-than-light travel. </p>\n<p>Despite being a fictional concept, various theoretical frameworks in physics suggest that such technology might be possible. Notably, ideas like wormholes, theorized by scientists, could potentially allow for shortcuts through space if harnessed correctly. In 1994,

In [ ]:
##loaddata--> docs ----> divide our text into chunks ---> embeddings ---> vectorstoredb

In [ ]:
## Splitting text into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

textsplitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=textsplitter.split_documents(res)

documents


[Document(metadata={'source': 'https://www.ebsco.com/research-starters/engineering/warp-drive', 'title': 'Warp drive | Engineering | Research Starters | EBSCO Research', 'description': '<p>Warp drive is a theoretical propulsion method that enables spacecraft to exceed the speed of light, primarily explored in science fiction literature and media. Originating in the 1920s and gaining prominence through the iconic television series Star Trek, the warp drive concept serves as a solution to the immense distances in the universe that hinder space travel. It proposes the creation of a &quot;warped&quot; bubble around a spacecraft, facilitated by a substantial power source, allowing it to navigate faster-than-light travel. </p>\n<p>Despite being a fictional concept, various theoretical frameworks in physics suggest that such technology might be possible. Notably, ideas like wormholes, theorized by scientists, could potentially allow for shortcuts through space if harnessed correctly. In 1994,

In [ ]:
#using openai embeddings

from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()


In [ ]:
#using faiss vector store database

from langchain_community.vectorstores import FAISS
vectordb=FAISS.from_documents(documents,embeddings)

In [ ]:
##query for testing 
query=" The fastest human-made object to pass beyond the boundary of the solar system was "
result=vectordb.similarity_search(query)

result[0].page_content

"is the distance that light—moving at a speed of about 186,000 miles per second—travels in one year's time. The closest star to Earth's sun is called Proxima Centauri, which is about 4.2 light years away. This means that light leaving Proxima Centauri takes 4.2 years to reach the Earth. The brightest star in the nighttime sky, Sirius, is about 8.6 light years away; the center of our galaxy, the Milky Way, is about 27,000 light years away; and the Andromeda Galaxy, our closest galactic neighbor, is about 2.5 million light years away.The fastest human-made object to pass beyond the boundary of the solar system was the Voyager 1 spacecraft, launched in 1977. At its current speed of 38,000 miles per hour, Voyager 1 would take more than 70,000 years to reach Proxima Centauri, and it is not even headed in that direction. Some theoretical projects, such as the nuclear fusion-powered Project Daedalus, could conceivably trim that time to a few decades, but human technology is nowhere near being

In [29]:
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(model="gpt-5.5")
print(llm)

output_version=None profile={'name': 'GPT-5.5', 'release_date': '2026-04-23', 'last_updated': '2026-04-23', 'open_weights': False, 'max_input_tokens': 1050000, 'max_output_tokens': 130000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True} client=<openai.resources.chat.completions.completions.Completions object at 0x0000020F910A6E00> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000020F910A64D0> root_client=<openai.OpenAI object at 0x0000020F910A7490> root_async_client=<openai.AsyncOpenAI object at 0x0000020F910A6560> model_name='gpt-5.5' model_kwargs={} openai_api_key=SecretStr(

In [ ]:
##Retrieval chain & document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     """
     Answer the question based on the following provided context:

     <context>
     {context}
     </context>
     """),
    ("human", "{input}")
])


document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n     Answer the question based on the following provided context:\n\n     <context>\n     {context}\n     </context>\n     '), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(output_version=None, profile={'name': 'GPT-5.5', 'release_date': '2026-04-23', 'last_updated': '2026-04-23', 'open_weights': False, 'max_input_tokens': 1050000, 'max_output_tokens': 130000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_

In [36]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"Einstein's theory leaves open the possibility that an extremely massive",
    "context": [Document(page_content="While the warp drive can be seen as a timeworn fictional plot device, some scientists have theorized that the technology is possible, even if beyond current human capability. Einstein's theory leaves open the possibility that an extremely massive object, such as a black hole (the remains of a collapsed star so dense that even light cannot escape it), could create tunnels between two sections of space. Other scientists have speculated that if a spacecraft could harness these wormholes, then it could conceivably bypass the vast interstellar distances and jump from one part of the universe to another. The problem arises that creating such a bridge requires a theoretical form of matter that does not adhere to the laws of physics. This exotic matter could stabilize a possible wormhole but could also expose astronauts to high radiation and other dangers.")]
    
})

'object, such as a black hole, could create tunnels between two sections of space.'

In [ ]:
## input--> Retrievers ---->>vectorstoredb 

retriever=vectordb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)    

 ##retriever contains the context and document chain contains the llm and 
 ##prompt template to answer the question based on the context provided by retriever


In [38]:
print(retrieval_chain)

bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020F910A5F00>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n     Answer the question based on the following provided context:\n\n     <context>\n     {context}\n     </context>\n     '), additional_kwargs={}), Hum

In [41]:
##get response from llm

response=retrieval_chain.invoke({
    "input":"What is the fastest human-made object to pass beyond the boundary of the solar system?"
})


print(response)

{'input': 'What is the fastest human-made object to pass beyond the boundary of the solar system?', 'context': [Document(id='21be6f0c-655e-40e8-9aa7-2a23bfcd304f', metadata={'source': 'https://www.ebsco.com/research-starters/engineering/warp-drive', 'title': 'Warp drive | Engineering | Research Starters | EBSCO Research', 'description': '<p>Warp drive is a theoretical propulsion method that enables spacecraft to exceed the speed of light, primarily explored in science fiction literature and media. Originating in the 1920s and gaining prominence through the iconic television series Star Trek, the warp drive concept serves as a solution to the immense distances in the universe that hinder space travel. It proposes the creation of a &quot;warped&quot; bubble around a spacecraft, facilitated by a substantial power source, allowing it to navigate faster-than-light travel. </p>\n<p>Despite being a fictional concept, various theoretical frameworks in physics suggest that such technology might

In [42]:
print(response['answer'])

The fastest human-made object to pass beyond the boundary of the solar system is the **Voyager 1 spacecraft**, launched in **1977**.


In [44]:
res2=retrieval_chain.invoke({
    "input":"Einstein's theory leaves open the possibility that an extremely massive"
})

res2['answer']


"Einstein's theory leaves open the possibility that an extremely massive object, such as a black hole, could create tunnels between two sections of space."